# Part 6: The Complete Transformer Architecture

## From Blocks to Full Models

Now we'll see how Transformer blocks are arranged into complete architectures:

1. **Encoder-Decoder** (Original Transformer) - For translation
2. **Encoder-only** (BERT) - For understanding
3. **Decoder-only** (GPT) - For generation

We'll focus on the **Decoder-only** architecture since that's what we're building for text generation.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


## Architecture Comparison

### Original Transformer (Encoder-Decoder)
- **Encoder**: Processes the input sequence
- **Decoder**: Generates the output, attending to encoder output
- **Use case**: Machine translation, summarization

### BERT (Encoder-only)
- **Bidirectional**: Can see all tokens
- **Use case**: Classification, question answering, understanding tasks

### GPT (Decoder-only) - What we're building!
- **Causal**: Can only see past tokens
- **Use case**: Text generation, completion, conversation


In [ ]:
# Visualize the three architectures
fig, axes = plt.subplots(1, 3, figsize=(16, 8))

def draw_block(ax, x, y, text, color='lightblue'):
    ax.add_patch(plt.Rectangle((x-0.15, y-0.08), 0.3, 0.16, 
                                facecolor=color, edgecolor='black'))
    ax.text(x, y, text, ha='center', va='center', fontsize=8)

# Encoder-Decoder
ax = axes[0]
# Encoder side
draw_block(ax, 0.2, 0, 'Input Emb', 'lightyellow')
for i in range(1, 4):
    draw_block(ax, 0.2, i*0.3, f'Enc {i}', 'lightgreen')
draw_block(ax, 0.2, 1.2, 'Enc Out', 'lightgreen')

# Decoder side  
draw_block(ax, 0.7, 0, 'Output Emb', 'lightyellow')
for i in range(1, 4):
    draw_block(ax, 0.7, i*0.3, f'Dec {i}', 'lightblue')
draw_block(ax, 0.7, 1.2, 'Linear', 'lightyellow')
draw_block(ax, 0.7, 1.5, 'Softmax', 'lightcoral')

# Cross attention arrows
for i in range(1, 4):
    ax.annotate('', xy=(0.55, i*0.3), xytext=(0.35, i*0.3),
                arrowprops=dict(arrowstyle='->', color='purple', lw=1))

ax.set_xlim(0, 1)
ax.set_ylim(-0.2, 1.7)
ax.axis('off')
ax.set_title('Encoder-Decoder\n(Translation)', fontsize=12)

# BERT (Encoder-only)
ax = axes[1]
draw_block(ax, 0.5, 0, 'Input Emb', 'lightyellow')
for i in range(1, 5):
    draw_block(ax, 0.5, i*0.3, f'Encoder {i}', 'lightgreen')
draw_block(ax, 0.5, 1.5, 'Task Head', 'lightcoral')

ax.set_xlim(0, 1)
ax.set_ylim(-0.2, 1.7)
ax.axis('off')
ax.set_title('Encoder-only (BERT)\n(Understanding)', fontsize=12)

# GPT (Decoder-only)
ax = axes[2]
draw_block(ax, 0.5, 0, 'Input Emb', 'lightyellow')
for i in range(1, 5):
    draw_block(ax, 0.5, i*0.3, f'Decoder {i}', 'lightblue')
draw_block(ax, 0.5, 1.5, 'LM Head', 'lightcoral')

# Causal indicator
ax.text(0.85, 0.6, 'Causal\nMask', ha='center', fontsize=8, color='red')

ax.set_xlim(0, 1)
ax.set_ylim(-0.2, 1.7)
ax.axis('off')
ax.set_title('Decoder-only (GPT)\n(Generation)', fontsize=12)

plt.tight_layout()
plt.show()


## The GPT Architecture

```
Input: "The cat sat"
         |
         v
  [Token Embedding]
         +
  [Position Embedding]
         |
         v
  [Decoder Block 1]  <-- Causal mask (can't see future)
         |
         v
  [Decoder Block 2]
         |
         v
       ...
         |
         v
  [Decoder Block N]
         |
         v
   [Final LayerNorm]
         |
         v
   [Language Model Head]
         |
         v
  Probability distribution over vocabulary
  (predict next token: "on")
```


In [ ]:
# Import our building blocks (recreating them here for completeness)

class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.eps = eps
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
    
    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta


class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = np.random.randn(d_model, d_model) * 0.02
        self.W_K = np.random.randn(d_model, d_model) * 0.02
        self.W_V = np.random.randn(d_model, d_model) * 0.02
        self.W_O = np.random.randn(d_model, d_model) * 0.02
    
    def forward(self, X, mask=None):
        seq_len = X.shape[0]
        
        Q = X @ self.W_Q
        K = X @ self.W_K
        V = X @ self.W_V
        
        Q = Q.reshape(seq_len, self.num_heads, self.d_k).transpose(1, 0, 2)
        K = K.reshape(seq_len, self.num_heads, self.d_k).transpose(1, 0, 2)
        V = V.reshape(seq_len, self.num_heads, self.d_k).transpose(1, 0, 2)
        
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)
        
        attn = softmax(scores, axis=-1)
        context = np.matmul(attn, V)
        context = context.transpose(1, 0, 2).reshape(seq_len, self.d_model)
        
        return context @ self.W_O, attn


class FeedForward:
    def __init__(self, d_model, d_ff=None):
        d_ff = d_ff or 4 * d_model
        self.W1 = np.random.randn(d_model, d_ff) * 0.02
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.02
        self.b2 = np.zeros(d_model)
    
    def forward(self, x):
        return gelu(x @ self.W1 + self.b1) @ self.W2 + self.b2


class DecoderBlock:
    """GPT-style decoder block with Pre-LN."""
    
    def __init__(self, d_model, num_heads, d_ff=None):
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Pre-LN style
        attn_out, attn_weights = self.attn.forward(self.norm1.forward(x), mask)
        x = x + attn_out
        
        ffn_out = self.ffn.forward(self.norm2.forward(x))
        x = x + ffn_out
        
        return x, attn_weights


print("Building blocks loaded!")


## The Complete GPT Model


In [ ]:
class GPT:
    """
    A GPT-style language model implemented in NumPy.
    
    This is a decoder-only transformer for text generation.
    """
    
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, d_ff=None):
        """
        vocab_size: Size of vocabulary
        d_model: Model dimension
        num_heads: Number of attention heads
        num_layers: Number of decoder blocks
        max_seq_len: Maximum sequence length
        d_ff: Feed-forward hidden dimension (default: 4 * d_model)
        """
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        # Token embeddings (vocab_size -> d_model)
        self.token_embedding = np.random.randn(vocab_size, d_model) * 0.02
        
        # Position embeddings (learned, like GPT)
        self.position_embedding = np.random.randn(max_seq_len, d_model) * 0.02
        
        # Decoder blocks
        self.blocks = [
            DecoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ]
        
        # Final layer norm
        self.final_norm = LayerNorm(d_model)
        
        # Language model head (projects back to vocabulary)
        # Often tied with token_embedding.T for parameter efficiency
        self.lm_head = np.random.randn(d_model, vocab_size) * 0.02
    
    def create_causal_mask(self, seq_len):
        """Create mask to prevent attending to future tokens."""
        return np.tril(np.ones((seq_len, seq_len)))
    
    def forward(self, token_ids):
        """
        Forward pass through the model.
        
        token_ids: List of token indices (seq_len,)
        Returns: logits (seq_len, vocab_size) - unnormalized predictions
        """
        seq_len = len(token_ids)
        
        # 1. Get embeddings
        x = self.token_embedding[token_ids]  # (seq_len, d_model)
        x = x + self.position_embedding[:seq_len]  # Add position info
        
        # 2. Create causal mask
        mask = self.create_causal_mask(seq_len)
        
        # 3. Pass through decoder blocks
        all_attentions = []
        for block in self.blocks:
            x, attn = block.forward(x, mask)
            all_attentions.append(attn)
        
        # 4. Final layer norm
        x = self.final_norm.forward(x)
        
        # 5. Project to vocabulary
        logits = x @ self.lm_head  # (seq_len, vocab_size)
        
        return logits, all_attentions
    
    def generate(self, start_tokens, max_new_tokens, temperature=1.0):
        """
        Generate text autoregressively.
        
        start_tokens: Initial token ids
        max_new_tokens: How many tokens to generate
        temperature: Higher = more random, lower = more deterministic
        """
        tokens = list(start_tokens)
        
        for _ in range(max_new_tokens):
            # Truncate if too long
            context = tokens[-self.max_seq_len:]
            
            # Get predictions
            logits, _ = self.forward(context)
            
            # Get logits for last position
            last_logits = logits[-1] / temperature
            
            # Convert to probabilities
            probs = softmax(last_logits)
            
            # Sample next token
            next_token = np.random.choice(len(probs), p=probs)
            tokens.append(next_token)
        
        return tokens


# Create a small GPT model
vocab_size = 100
d_model = 64
num_heads = 4
num_layers = 4
max_seq_len = 128

gpt = GPT(vocab_size, d_model, num_heads, num_layers, max_seq_len)

# Test forward pass
test_tokens = [1, 5, 10, 15, 20]
logits, attentions = gpt.forward(test_tokens)

print("GPT Model Created!")
print(f"  Vocab size: {vocab_size}")
print(f"  Model dimension: {d_model}")
print(f"  Num heads: {num_heads}")
print(f"  Num layers: {num_layers}")
print(f"  Max sequence length: {max_seq_len}")
print(f"\nForward pass:")
print(f"  Input: {len(test_tokens)} tokens")
print(f"  Output logits shape: {logits.shape}")
print(f"  Attention maps: {len(attentions)} layers")


In [ ]:
# Visualize the generation process
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Logits at last position
ax = axes[0, 0]
last_logits = logits[-1]
ax.bar(range(min(30, vocab_size)), last_logits[:30])
ax.set_xlabel('Token ID')
ax.set_ylabel('Logit Value')
ax.set_title('Logits at Last Position\n(First 30 tokens)')
ax.grid(True, alpha=0.3)

# 2. Probabilities after softmax
ax = axes[0, 1]
probs = softmax(last_logits)
ax.bar(range(min(30, vocab_size)), probs[:30])
ax.set_xlabel('Token ID')
ax.set_ylabel('Probability')
ax.set_title('Probabilities After Softmax\n(First 30 tokens)')
ax.grid(True, alpha=0.3)

# 3. Effect of temperature
ax = axes[1, 0]
for temp in [0.5, 1.0, 2.0]:
    probs_temp = softmax(last_logits / temp)
    ax.plot(range(min(30, vocab_size)), probs_temp[:30], label=f'T={temp}', lw=2)
ax.set_xlabel('Token ID')
ax.set_ylabel('Probability')
ax.set_title('Effect of Temperature\n(Lower = more peaked)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Causal mask visualization
ax = axes[1, 1]
mask = gpt.create_causal_mask(8)
im = ax.imshow(mask, cmap='Greens')
ax.set_xlabel('Attending TO')
ax.set_ylabel('Attending FROM')
ax.set_title('Causal Mask\n(1=allowed, 0=blocked)')
for i in range(8):
    for j in range(8):
        ax.text(j, i, int(mask[i,j]), ha='center', va='center')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()


## Text Generation Demo

Let's generate some random tokens (the model is untrained, so it won't make sense!):


In [ ]:
# Generate some tokens
start_tokens = [1, 2, 3]  # Starting context
generated = gpt.generate(start_tokens, max_new_tokens=20, temperature=1.0)

print("Generation Demo (random, untrained model):")
print("=" * 50)
print(f"Start tokens:     {start_tokens}")
print(f"Generated tokens: {generated}")
print(f"New tokens:       {generated[len(start_tokens):]}")

# Compare different temperatures
print("\nEffect of temperature:")
for temp in [0.5, 1.0, 2.0]:
    np.random.seed(42)  # Same seed for comparison
    gen = gpt.generate([1, 2, 3], max_new_tokens=10, temperature=temp)
    print(f"  T={temp}: {gen[3:]}")


## Model Size Calculation

Let's count the parameters:


In [ ]:
def count_parameters(model):
    """Count total parameters in our GPT model."""
    total = 0
    
    # Token embeddings
    token_emb = model.vocab_size * model.d_model
    total += token_emb
    print(f"Token embeddings: {token_emb:,}")
    
    # Position embeddings
    pos_emb = model.max_seq_len * model.d_model
    total += pos_emb
    print(f"Position embeddings: {pos_emb:,}")
    
    # Each decoder block
    d = model.d_model
    d_ff = 4 * d
    
    per_block = (
        4 * d * d +  # W_Q, W_K, W_V, W_O
        2 * d * d_ff +  # FFN W1, W2
        d_ff + d +  # FFN biases
        4 * d  # LayerNorm gammas and betas (2 norms per block)
    )
    
    blocks_total = per_block * len(model.blocks)
    total += blocks_total
    print(f"Decoder blocks ({len(model.blocks)}x): {blocks_total:,}")
    
    # Final layer norm
    final_ln = 2 * d
    total += final_ln
    print(f"Final LayerNorm: {final_ln:,}")
    
    # LM head
    lm_head = d * model.vocab_size
    total += lm_head
    print(f"LM head: {lm_head:,}")
    
    print(f"\nTotal parameters: {total:,}")
    return total

print("Parameter count for our model:")
print("=" * 50)
params = count_parameters(gpt)

# Compare to real models
print("\n\nComparison to real models:")
print("=" * 50)
models = {
    "Our model": params,
    "GPT-2 Small": 124_000_000,
    "GPT-2 Medium": 355_000_000,
    "GPT-2 Large": 774_000_000,
    "GPT-3": 175_000_000_000,
}

for name, p in models.items():
    print(f"{name:15} {p:>15,} parameters")


## Summary: Complete GPT Architecture

### Components (in order)

1. **Token Embedding**: vocab_size -> d_model
2. **Position Embedding**: Added to token embeddings
3. **N x Decoder Blocks**: Each containing:
   - Layer Norm
   - Multi-Head Self-Attention (with causal mask)
   - Residual connection
   - Layer Norm
   - Feed-Forward Network
   - Residual connection
4. **Final Layer Norm**
5. **LM Head**: d_model -> vocab_size

### Key Design Choices

| Choice | Original Transformer | GPT |
|--------|---------------------|-----|
| Normalization | Post-LN | Pre-LN |
| Position encoding | Sinusoidal (fixed) | Learned |
| Attention | Bidirectional | Causal only |
| Architecture | Encoder-Decoder | Decoder only |

### Training Objective

**Next token prediction**: Given tokens 1...t, predict token t+1

```
Input:  "The cat sat on the"
Target: "cat sat on the mat"
```

Each position predicts its next token!

---

**Next: 07_text_generation.ipynb** - Train and generate real text with PyTorch!
